In [43]:
import os
import json
import time
import pathlib
from string import Template
from dotenv import load_dotenv
from openai import OpenAI
from typing import List, Dict


In [44]:
BASE_DIR = pathlib.Path.cwd()

load_dotenv(BASE_DIR / "environment.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY not set in .env")


In [45]:
with open(BASE_DIR / "config.json", "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

OPENAI_CFG = CONFIG["openai"]
OUTPUT_CFG = CONFIG["output"]


In [46]:
client = OpenAI(api_key=OPENAI_API_KEY)


In [47]:
OUTPUT_DIR = BASE_DIR / OUTPUT_CFG.get("output_dir")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [48]:
RETRIEVAL_RESULTS_PATH = pathlib.Path(
    CONFIG.get("retrieval", {}).get("results_path", BASE_DIR / "retrieval_outputs/retrieval_results.jsonl")
)


In [49]:
with open(BASE_DIR / "toulmin_prompt.txt", "r", encoding="utf-8") as f:
    TOULMIN_TEMPLATE = Template(f.read())


In [50]:
def build_toulmin_prompt(claim_text: str, article: Dict) -> str:
    return TOULMIN_TEMPLATE.substitute(
        claim_text=claim_text,
        pubmed_id=article.get("pmid") or article.get("pubmed_id", ""),
        title=article.get("title", ""),
        abstract=article.get("abstract", "")
    )


In [51]:
def extract_toulmin_argument(article: Dict, claim_text: str) -> Dict:
    prompt = build_toulmin_prompt(claim_text, article)

    response = client.chat.completions.create(
        model=OPENAI_CFG["model"],
        temperature=OPENAI_CFG.get("temperature", 0),
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": "You are a precise biomedical argument extraction assistant."},
            {"role": "user", "content": prompt}
        ]
    )

    content = response.choices[0].message.content
    try:
        parsed = json.loads(content)
    except json.JSONDecodeError:
        parsed = {
            "stance": "parse_error",
            "toulmin": {
                "claim": "",
                "data": [],
                "warrant": "",
                "backing": [],
                "qualifier": "",
                "rebuttals": [],
                "raw": content
            }
        }

    return parsed


In [52]:
def run_pipeline():
    if not RETRIEVAL_RESULTS_PATH.exists():
        print(f"Missing retrieval results: {RETRIEVAL_RESULTS_PATH}")
        return

    with RETRIEVAL_RESULTS_PATH.open("r", encoding="utf-8") as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    if not lines:
        print("No retrieval records found.")
        return

    for line in lines:
        rec = json.loads(line)
        claim_id = rec.get("claim_id", "retrieval")
        claim_text = rec.get("query", "")
        articles = rec.get("articles", [])

        print(f"=== Processing retrieval record {claim_id} ===")
        print(f"Query: {claim_text}")
        print(f"Total articles to process with LLM: {len(articles)}")

        out_path = OUTPUT_DIR / f"{claim_id}_toulmin.jsonl"
        with out_path.open("a", encoding="utf-8") as f_out:
            for idx, art in enumerate(articles, start=1):
                pmid = art.get("pmid") or art.get("pubmed_id")
                print(f"[{claim_id}] [{idx}/{len(articles)}] PMID {pmid}")
                toulmin_arg = extract_toulmin_argument(art, claim_text)

                record = {
                    "claim_id": claim_id,
                    "claim_text": claim_text,
                    "source": {
                        "pubmed_id": pmid,
                        "title": art.get("title"),
                        "year": art.get("year"),
                        "journal": art.get("journal"),
                        "evidence_type": art.get("_evidence_type"),
                        "abstract": art.get("abstract")
                    },
                    "stance": toulmin_arg.get("stance"),
                    "toulmin": toulmin_arg.get("toulmin", {})
                }

                f_out.write(json.dumps(record, ensure_ascii=False) + "\n")
                time.sleep(0.5)

        print(f"Saved Toulmin arguments for {claim_id} to {out_path}")


In [53]:
run_pipeline()


=== Processing retrieval record retrieval ===
Query: Semaglutide induces significant weight loss in adults with obesity.
Total articles to process with LLM: 15
[retrieval] [1/15] PMID 36254579


2026-03-01 20:44:49,305 INFO httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


[retrieval] [2/15] PMID 40125230


2026-03-01 20:44:55,410 INFO httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


[retrieval] [3/15] PMID 34942372


...

[retrieval] [14/15] PMID 35226299


2026-03-01 20:45:52,028 INFO httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


[retrieval] [15/15] PMID 36569429


2026-03-01 20:46:02,681 INFO httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Saved Toulmin arguments for retrieval to /Users/ftzavellos/Law_and_Tech/drug_explanations/Code/outputs/retrieval_toulmin.jsonl
